# Part 1 - SQL

This notebook contains my Part 1 solution using the public `bigquery-public-data.thelook_ecommerce` dataset.

It includes the SQL for Tasks A-D, the executed outputs, and the business definitions used throughout the analysis.

Notebook sections:
- Task A: monthly product and conversion metrics
- Task B: new vs returning customer and revenue mix
- Task C: 90-day churn analysis
- Task D: free shipping over $100 experiment support

To rerun the notebook, replace the Project ID in the next cell with your own Google Cloud Project ID.


In [ ]:
# INITIALIZATION
from google.cloud import bigquery
import pandas as pd

# Replace with your Project ID from your Google Cloud Console
PROJECT_ID = "your-project-id"  
client = bigquery.Client(project=PROJECT_ID)

print("BigQuery Client Initialized Successfully!")

/Users/roger/Documents/thelook_ecommerce/.venv/lib/python3.14/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


BigQuery Client Initialized Successfully!


In [ ]:
# ==========================================
# TASK A - MONTHLY METRICS
# ==========================================
sql_task_a = """
/*
===========================================================================
TASK A: MONTHLY PRODUCT METRICS
===========================================================================
ASSUMPTIONS & DEFINITIONS:
1. Conversion Rate Proxy: Defined as:
   (Distinct users with 'purchase' event in month X) / 
   (Distinct users with 'department' event in month X)
   Limitation: This is a monthly user-level proxy. It does not guarantee 
   intra-month sequence (a user might buy, then browse a department later) 
   and is not strictly session-based.
===========================================================================
*/

WITH query_params AS (
    -- Set the start date here so it is fully parameterized
    SELECT DATE('2020-01-01') AS start_date
),
sales_metrics AS (
    SELECT
        DATE_TRUNC(DATE(oi.created_at), MONTH) AS month,
        SUM(oi.sale_price) AS revenue,
        COUNT(DISTINCT oi.order_id) AS orders,
        COUNT(oi.id) AS units,
        SAFE_DIVIDE(SUM(oi.sale_price), COUNT(DISTINCT oi.order_id)) AS aov
    FROM `bigquery-public-data.thelook_ecommerce.order_items` oi
    CROSS JOIN query_params qp
    WHERE oi.status = 'Complete' AND oi.returned_at IS NULL
      AND DATE(oi.created_at) >= qp.start_date
    GROUP BY 1
),
funnel_metrics AS (
    SELECT
        DATE_TRUNC(DATE(e.created_at), MONTH) AS month,
        COUNT(DISTINCT CASE WHEN e.event_type = 'department' THEN e.user_id END) AS top_funnel_users,
        COUNT(DISTINCT CASE WHEN e.event_type = 'purchase' THEN e.user_id END) AS converted_users
    FROM `bigquery-public-data.thelook_ecommerce.events` e
    CROSS JOIN query_params qp
    WHERE DATE(e.created_at) >= qp.start_date
    GROUP BY 1
)

SELECT 
    s.month,
    s.revenue,
    s.orders,
    s.units,
    s.aov,
    SAFE_DIVIDE(f.converted_users, f.top_funnel_users) AS monthly_conversion_rate
FROM sales_metrics s
LEFT JOIN funnel_metrics f ON s.month = f.month
ORDER BY s.month DESC;
"""

df_task_a = client.query(sql_task_a).to_dataframe()
display(df_task_a.head())

,month,revenue,orders,units,aov,monthly_conversion_rate
0,2026-04-01,115998.070135,1318,1908,88.010675,1.055571
1,2026-03-01,153266.630110,1824,2613,84.027758,1.014345
2,2026-02-01,110072.380183,1279,1815,86.061282,1.014858
3,2026-01-01,103010.610086,1259,1775,81.819388,1.014292
4,2025-12-01,102181.300097,1188,1712,86.011195,1.018202


In [ ]:
# ==========================================
#  TASK B - NEW VS RETURNING MIX
# ==========================================
sql_task_b = """
/*
===========================================================================
TASK B: NEW VS RETURNING MIX
===========================================================================
DEFINITIONS:
1. Active Customer: A customer with >= 1 completed order in that month.
2. New Customer: A customer whose first-ever completed order occurs in 
   that specific month.
3. Returning Customer: A customer who makes a completed order in a month 
   strictly AFTER their first-ever completed order month.
===========================================================================
*/

WITH user_activation AS (
    SELECT
        user_id,
        DATE_TRUNC(DATE(MIN(created_at)), MONTH) AS first_purchase_month
    FROM `bigquery-public-data.thelook_ecommerce.order_items`
    WHERE status = 'Complete' AND returned_at IS NULL
    GROUP BY 1
),
monthly_user_sales AS (
    SELECT
        o.user_id,
        DATE_TRUNC(DATE(o.created_at), MONTH) AS order_month,
        SUM(o.sale_price) AS user_revenue
    FROM `bigquery-public-data.thelook_ecommerce.order_items` o
    WHERE o.status = 'Complete' AND o.returned_at IS NULL
    GROUP BY 1, 2
)

SELECT
    m.order_month AS month,
    COUNT(DISTINCT m.user_id) AS active_customers,
    
    COUNT(DISTINCT CASE WHEN m.order_month = a.first_purchase_month THEN m.user_id END) AS new_customers,
    COUNT(DISTINCT CASE WHEN m.order_month > a.first_purchase_month THEN m.user_id END) AS returning_customers,
    
    SUM(CASE WHEN m.order_month = a.first_purchase_month THEN m.user_revenue ELSE 0 END) AS revenue_new,
    SUM(CASE WHEN m.order_month > a.first_purchase_month THEN m.user_revenue ELSE 0 END) AS revenue_returning,
    
    SAFE_DIVIDE(
        SUM(CASE WHEN m.order_month > a.first_purchase_month THEN m.user_revenue ELSE 0 END),
        SUM(m.user_revenue)
    ) AS pct_revenue_from_returning

FROM monthly_user_sales m
JOIN user_activation a ON m.user_id = a.user_id
GROUP BY 1
ORDER BY 1 DESC;
"""

df_task_b = client.query(sql_task_b).to_dataframe()
display(df_task_b.head())

,month,active_customers,new_customers,returning_customers,revenue_new,revenue_returning,pct_revenue_from_returning
0,2026-04-01,1206,1083,123,105338.620108,10659.450027,0.091893
1,2026-03-01,1766,1462,304,128616.560077,24650.070033,0.160831
2,2026-02-01,1262,1031,231,88754.570153,21317.810030,0.193671
3,2026-01-01,1247,1058,189,86989.520055,16021.090032,0.155529
4,2025-12-01,1178,1000,178,89028.220103,13153.079995,0.128723


In [ ]:
# ==========================================
# TASK C - 90-DAY CHURN
# ==========================================
sql_task_c = """
/*
===========================================================================
TASK C: 90-DAY CHURN
===========================================================================
DEFINITIONS & LIMITATIONS:
1. Churn Definition: A user is considered churned for a given active month 
   if they have ZERO completed orders in the 90 days following their LAST 
   order in that specific month.
2. Reproducibility / Dataset Boundary: To ensure historical accuracy, we 
   calculate the global MAX(created_at) of the dataset rather than using 
   CURRENT_DATE(). This safely censors recent months that have not had a 
   full 90 days to mature within the dataset boundaries.
3. Real-World Refinement (Limitation): 90-day purchase churn is a lagging 
   indicator. In a real setting, measuring "session churn" (e.g., 14 days 
   without an app login) enables much faster proactive intervention.
===========================================================================
*/

WITH dataset_bounds AS (
    -- Dynamically find the latest date in the dataset to ensure reproducibility
    SELECT MAX(DATE(created_at)) AS global_max_date
    FROM `bigquery-public-data.thelook_ecommerce.order_items`
),
user_active_months AS (
    SELECT
        user_id,
        DATE_TRUNC(DATE(created_at), MONTH) AS active_month,
        MAX(DATE(created_at)) AS last_order_date_in_month
    FROM `bigquery-public-data.thelook_ecommerce.order_items`
    WHERE status = 'Complete' AND returned_at IS NULL
    GROUP BY 1, 2
),
user_future_activity AS (
    SELECT
        u.user_id,
        u.active_month,
        -- Boolean check is cleaner than counting rows
        MAX(CASE WHEN n.order_id IS NOT NULL THEN 1 ELSE 0 END) AS has_future_order
    FROM user_active_months u
    CROSS JOIN dataset_bounds b
    LEFT JOIN `bigquery-public-data.thelook_ecommerce.order_items` n
        ON u.user_id = n.user_id
        AND n.status = 'Complete' AND n.returned_at IS NULL
        AND DATE(n.created_at) > u.last_order_date_in_month
        AND DATE(n.created_at) <= DATE_ADD(u.last_order_date_in_month, INTERVAL 90 DAY)
    -- Crucial Fix: Use dataset boundary instead of CURRENT_DATE()
    WHERE DATE_ADD(u.last_order_date_in_month, INTERVAL 90 DAY) <= b.global_max_date
    GROUP BY 1, 2
)

SELECT
    active_month AS month,
    COUNT(DISTINCT user_id) AS active_customers,
    COUNT(DISTINCT CASE WHEN has_future_order = 0 THEN user_id END) AS churned_customers_90d,
    SAFE_DIVIDE(
        COUNT(DISTINCT CASE WHEN has_future_order = 0 THEN user_id END),
        COUNT(DISTINCT user_id)
    ) AS churn_rate_90d
FROM user_future_activity
GROUP BY 1
ORDER BY 1 DESC;
"""

df_task_c = client.query(sql_task_c).to_dataframe()
display(df_task_c.head())

,month,active_customers,churned_customers_90d,churn_rate_90d
0,2026-01-01,395,371,0.939241
1,2025-12-01,1178,1116,0.947368
2,2025-11-01,1030,964,0.935922
3,2025-10-01,1010,956,0.946535
4,2025-09-01,893,846,0.947368


In [ ]:
# ==========================================
# TASK D - SCENARIO IMPACT
# ==========================================
sql_task_d = """
/*
===========================================================================
TASK D: PRODUCT CHANGE IMPACT ANALYSIS (FREE SHIPPING > $100)
===========================================================================
ASSUMPTIONS:
1. Feature Rollout: Assumes 100% rollout on launch date.
2. Cart Value Proxy: Assumes SUM(sale_price) per order accurately reflects 
   the pre-tax cart value determining shipping eligibility.
3. Seasonality: Assumes the 30-day post-launch window (Jan/Feb) does not 
   have severe seasonal variance compared to the 30-day pre-launch window.
4. Parameterization: Date windows are isolated in a CTE for clean testing.

ADDITIONAL DATA NEEDED FOR A REAL ENVIRONMENT:
1. Experiment Assignments: Variant tables (Control vs Treatment) to measure 
   causal lift instead of a pre/post proxy.
2. Shipping Financials: A 'shipping_fee' field to calculate true margin loss.
===========================================================================
*/

WITH scenario_params AS (
    -- Parameterized date logic
    SELECT 
        DATE('2022-01-15') AS launch_date,
        DATE('2021-12-16') AS pre_window_start,
        DATE('2022-02-14') AS post_window_end
),
cart_totals AS (
    SELECT
        oi.order_id,
        u.traffic_source,
        DATE_TRUNC(DATE(oi.created_at), MONTH) AS order_month,
        DATE(oi.created_at) AS order_date,
        SUM(oi.sale_price) AS total_cart_value
    FROM `bigquery-public-data.thelook_ecommerce.order_items` oi
    JOIN `bigquery-public-data.thelook_ecommerce.users` u 
        ON oi.user_id = u.id
    CROSS JOIN scenario_params p
    WHERE oi.status = 'Complete' AND oi.returned_at IS NULL
      AND DATE(oi.created_at) BETWEEN p.pre_window_start AND p.post_window_end
    GROUP BY 1, 2, 3, 4
),
impact_segments AS (
    SELECT 
        order_id,
        order_month,
        traffic_source,
        total_cart_value,
        CASE
            WHEN order_date < (SELECT launch_date FROM scenario_params) THEN '1. Pre-Launch'
            ELSE '2. Post-Launch'
        END AS period,
        CASE
            WHEN total_cart_value >= 100 THEN 'High Value (>= $100)'
            ELSE 'Low Value (< $100)'
        END AS cart_segment
    FROM cart_totals
)

SELECT
    order_month,
    period,
    traffic_source,
    cart_segment,
    COUNT(DISTINCT order_id) AS total_completed_orders,
    SUM(total_cart_value) AS total_revenue,
    SAFE_DIVIDE(SUM(total_cart_value), COUNT(DISTINCT order_id)) AS aov
FROM impact_segments
GROUP BY 1, 2, 3, 4
ORDER BY order_month, period, traffic_source, cart_segment;
"""

df_task_d = client.query(sql_task_d).to_dataframe()
display(df_task_d)

,order_month,period,traffic_source,cart_segment,total_completed_orders,total_revenue,aov
0,2021-12-01,1. Pre-Launch,Display,Low Value (< $100),3,75.440000,25.146667
1,2021-12-01,1. Pre-Launch,Email,Low Value (< $100),3,297.490002,99.163334
2,2021-12-01,1. Pre-Launch,Facebook,Low Value (< $100),4,231.579998,57.895000
3,2021-12-01,1. Pre-Launch,Organic,High Value (>= $100),2,273.950001,136.975000
4,2021-12-01,1. Pre-Launch,Organic,Low Value (< $100),8,294.279995,36.784999
5,2021-12-01,1. Pre-Launch,Search,High Value (>= $100),8,1908.280003,238.535000
6,2021-12-01,1. Pre-Launch,Search,Low Value (< $100),64,3197.220006,49.956563
7,2022-01-01,1. Pre-Launch,Display,High Value (>= $100),1,137.990002,137.990002
8,2022-01-01,1. Pre-Launch,Display,Low Value (< $100),4,135.990002,33.997500
9,2022-01-01,1. Pre-Launch,Email,High Value (>= $100),3,804.500000,268.166667
